<a href="https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold3_of3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AlphaFold 3 (OpenFold3 Preview 2 weights)

Predict protein, RNA, DNA, and small-molecule structures using [AlphaFold 3](https://www.nature.com/articles/s41586-024-07487-w) with the freely available [OpenFold3](https://github.com/aqlaboratory/openfold) weights (Apache 2.0 — no commercial restrictions).

MSA generation via the [ColabFold](https://github.com/sokrypton/ColabFold) MMseqs2 server — **no local databases required**.

Just fill in the **Input sequence(s)** cell and run all. Proteins, nucleic acids and ligands all go in a single box; attention/XLA flags are chosen automatically for your runtime (T4, newer GPUs, or CPU).

**Citations:**
- Abramson et al. (2024) AlphaFold 3. *Nature* [doi:10.1038/s41586-024-07487-w](https://doi.org/10.1038/s41586-024-07487-w)
- Mirdita et al. (2022) ColabFold. *Nature Methods* [doi:10.1038/s41592-022-01488-1](https://doi.org/10.1038/s41592-022-01488-1)
- The OpenFold3 Team (2025) [Github](https://github.com/aqlaboratory/openfold-3) - [doi:10.5281/zenodo.19001000](https://doi.org/10.5281/zenodo.19001000)

**Credits:** AF3 code: Google DeepMind (Apache 2.0) · OF3 weights: AlQuraishi Lab (Apache 2.0)

In [ ]:
#@title Install dependencies (~3 mins)
%%time
import os, time

if not os.path.isfile('ALPHAFOLD3_READY'):
  print('Installing packages...')
  os.system("pip install -q 'jax[cuda12]==0.10.1' dm-haiku==0.0.16 rdkit==2025.9.4 \
  zstandard awscli tokamax==0.0.11 py3Dmol")
  os.system("pip install -q --no-deps 'https://github.com/sokrypton/alphafold3/releases/download/v3.1.0/alphafold3_open-3.1.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl'")
  base = 'https://raw.githubusercontent.com/sokrypton/alphafold3/refs/heads/main'
  for script in ['run_alphafold', 'convert_of3_weights']:
    os.system(f'wget -q -nc {base}/{script}.py')
  os.system('touch ALPHAFOLD3_READY')
  print('Packages installed.')

# Download + convert OF3 weights (background)
if not os.path.isfile('WEIGHTS_DONE'):
  print('Downloading OpenFold3 weights...')
  # Wrapped in parentheses to safely background the whole chain
  os.system('(aws s3 cp s3://openfold/staging/of3-p2-155k.pt . --no-sign-request; \
  python convert_of3_weights.py --of3_checkpoint of3-p2-155k.pt --output_dir af3_converted_weights; \
  touch WEIGHTS_DONE) &')

# Build AF3 data files (background, independent of weights)
if not os.path.isfile('DATA_DONE'):
  print('Building AF3 data files...')
  os.system('(build_data; touch DATA_DONE) &')

# Wait for both
for sentinel in ['WEIGHTS_DONE', 'DATA_DONE']:
  while not os.path.isfile(sentinel):
    time.sleep(5)
  print(f'{sentinel} ✓')

print('Setup complete!')

In [ ]:
#@title Input sequences
import re, os, json, hashlib

#@markdown ### Molecules
#@markdown Separate multiple chains within a box using `:` (extra colons are fine: `A::::B` == `A:B`). Leave a box empty if unused; full details in the Instructions cell.
protein = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK' #@param {type:"string"}
dna = '' #@param {type:"string"}
rna = '' #@param {type:"string"}
ligand_ccd = '' #@param {type:"string"}
ligand_smiles = '' #@param {type:"string"}

#@markdown ### Run settings
jobname = 'test' #@param {type:"string"}
msa_mode = "mmseqs2_server" #@param ["mmseqs2_server", "single_sequence"]
seeds = '1' #@param {type:"string"}
#@markdown - `msa_mode`: `single_sequence` skips the MSA (faster, lower accuracy).
#@markdown - `seeds`: comma-separated, e.g. `1,2,3`.

# ── Split a box into entries: collapse colon runs, drop whitespace, skip empties ─
def split_entries(s):
  s = re.sub(r':+', ':', s).strip(':')
  return [e for e in (''.join(tok.split()) for tok in s.split(':')) if e]

prot_seqs   = [e.upper() for e in split_entries(protein)]
dna_seqs    = [e.upper() for e in split_entries(dna)]
rna_seqs    = [e.upper() for e in split_entries(rna)]
ccd_codes   = [e.upper() for e in split_entries(ligand_ccd)]
smiles_strs = split_entries(ligand_smiles)         # case-sensitive: leave as typed

# ── Build AF3 chain entities (IDs assigned A, B, C, ... in order) ─────────
CHAIN_IDS = list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz')
chains, prot_groups, idx = [], {}, 0

for seq in prot_seqs:
  cid = CHAIN_IDS[idx]; idx += 1
  if seq in prot_groups:                           # merge identical seqs -> homo-oligomer
    ent = prot_groups[seq]
    ids = ent['id'] if isinstance(ent['id'], list) else [ent['id']]
    ent['id'] = ids + [cid]
  else:
    ent = {'id': cid, 'sequence': seq, 'templates': []}
    if msa_mode == 'single_sequence':
      ent.update({'unpairedMsa': f'>query\n{seq}\n', 'pairedMsa': ''})
    prot_groups[seq] = ent
    chains.append({'protein': ent})

for seq in rna_seqs:
  c = {'id': CHAIN_IDS[idx], 'sequence': seq}
  if msa_mode == 'single_sequence':
    c['unpairedMsa'] = f'>query\n{seq}\n'
  chains.append({'rna': c}); idx += 1

for seq in dna_seqs:
  chains.append({'dna': {'id': CHAIN_IDS[idx], 'sequence': seq}}); idx += 1

for code in ccd_codes:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'ccdCodes': [code]}}); idx += 1

for smiles in smiles_strs:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'smiles': smiles}}); idx += 1

if not chains:
  raise ValueError('No valid input found — fill in at least one box.')

# ── Seeds: pull out integers regardless of separators, dedupe, default to [1] ─
seed_list = []
for tok in re.findall(r'\d+', seeds):
  v = int(tok)
  if v not in seed_list:
    seed_list.append(v)
if not seed_list:
  seed_list = [1]

# ── Job name (stable, derived from the inputs) ─────────────────────
def _flat(mol):
  if 'sequence' in mol: return mol['sequence']
  if 'ccdCodes' in mol: return ','.join(mol['ccdCodes'])
  return mol.get('smiles', '?')
flat = ':'.join(_flat(list(c.values())[0]) for c in chains)
basejob = re.sub(r'\W+', '', ''.join(jobname.split())) or 'job'
jobname = basejob + '_' + hashlib.sha1(flat.encode()).hexdigest()[:5]

def _next_free(name):
  if not os.path.exists(name): return name
  n = 0
  while os.path.exists(f'{name}_{n}'): n += 1
  return f'{name}_{n}'
jobname = _next_free(jobname)

OUTPUT_DIR = 'af3_output'
fold_input = {
    'name': jobname,
    'sequences': chains,
    'modelSeeds': seed_list,
    'dialect': 'alphafold3',
    'version': 1,
}
os.makedirs(jobname, exist_ok=True)
json_path = f'{jobname}/{jobname}.json'
with open(json_path, 'w') as f:
  json.dump(fold_input, f, indent=2)

fold_input


In [ ]:
#@title Run AlphaFold 3
%%time
import os, glob, subprocess

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Pick attention impl + XLA flags from the actual device ────────────
# Triton / cuDNN flash attention need Ampere (compute capability >= 8.0).
#   * 7.x GPUs (T4 = 7.5, V100 = 7.0): require flash=xla AND an XLA flag that
#     disables the custom-kernel fusion pass (run_alphafold.py errors otherwise).
#   * No GPU (CPU): also use flash=xla (the portable, no-flash path).
def detect_device():
  try:
    out = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15)
    caps = [float(x) for x in out.stdout.split() if x.strip()]
    if caps:
      return 'gpu', min(caps)
  except Exception:
    pass
  return 'cpu', None

device, cap = detect_device()
XLA_FIX = '--xla_disable_hlo_passes=custom-kernel-fusion-rewriter'
nojit = False

if device == 'cpu':
  flash_impl = 'xla'
  nojit = True   # skip the (very slow) whole-graph XLA compile; run eagerly instead
  print('No GPU detected — running on CPU with XLA attention + --nojit (slow, but avoids the compile).')
elif cap < 8.0:
  flash_impl = 'xla'
  cur = os.environ.get('XLA_FLAGS', '')           # set on os.environ so the child inherits it
  if XLA_FIX not in cur:
    os.environ['XLA_FLAGS'] = (cur + ' ' + XLA_FIX).strip()
  print(f'Pre-Ampere GPU (compute capability {cap}) — using XLA attention + XLA fusion fix.')
else:
  flash_impl = 'triton'
  print(f'Ampere+ GPU (compute capability {cap}) — using fast Triton flash attention.')

print('XLA_FLAGS =', os.environ.get('XLA_FLAGS', '(unset)'))

cmd = [
    'python', 'run_alphafold.py',
    f'--json_path={json_path}',
    '--model_dir=af3_converted_weights',
    '--of3_weights',
    '--norun_data_pipeline',
    f'--output_dir={OUTPUT_DIR}',
    '--cache_dir=/tmp/af3_cache',
    f'--flash_attention_implementation={flash_impl}',
]
if msa_mode == 'mmseqs2_server':
  cmd.append('--use_msa_server')
if nojit:
  cmd.append('--nojit')

cmd = ' '.join(cmd)
print(cmd)
!{cmd}

In [ ]:
#@title Display 3D structure {run: "auto"}
import csv, glob, os
import py3Dmol

rank_num = 1 #@param [1,2,3,4,5] {type:"raw"}
color = "pLDDT" #@param ["pLDDT", "chain", "rainbow"]
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}

# Rank 1 -> best model CIF written to the top-level output dir
# Rank 2+ -> look up in the ranking CSV
def get_cif_path(rank):
  if rank == 1:
    p = f'{OUTPUT_DIR}/{jobname}/{jobname}.cif'
    if os.path.exists(p):
      return p
  ranking_csv = f'{OUTPUT_DIR}/{jobname}/{jobname}_ranking_scores.csv'
  if os.path.exists(ranking_csv):
    rows = []
    with open(ranking_csv) as f:
      for row in csv.DictReader(f):
        rows.append((float(row['ranking_score']), int(row['seed']), int(row['sample'])))
    rows.sort(reverse=True)
    if rank <= len(rows):
      _, seed, sample = rows[rank - 1]
      return (f'{OUTPUT_DIR}/{jobname}/seed-{seed}_sample-{sample}/'
              f'{jobname}_seed-{seed}_sample-{sample}_model.cif')
  # Fallback
  cifs = sorted(glob.glob(f'{OUTPUT_DIR}/{jobname}/**/*.cif', recursive=True))
  return cifs[rank - 1] if rank <= len(cifs) else None

cif_path = get_cif_path(rank_num)
if not cif_path or not os.path.exists(cif_path):
  raise FileNotFoundError(f'Rank {rank_num} CIF not found in {OUTPUT_DIR}/{jobname}/')
print(f'Displaying: {cif_path}')

cif_str = open(cif_path).read()
view = py3Dmol.view(width=800, height=500, js='https://3dmol.org/build/3Dmol.js')
view.addModel(cif_str, 'cif')

if color == 'pLDDT':
  view.setStyle({'cartoon': {'colorscheme': {'prop': 'b', 'gradient': 'roygb', 'min': 50, 'max': 90}}})
elif color == 'rainbow':
  view.setStyle({'cartoon': {'color': 'spectrum'}})
elif color == 'chain':
  palette = ['cyan', 'magenta', 'yellow', 'salmon', 'lime',
             'slate', 'orange', 'deepsalmon', 'lightteal', 'yelloworange']
  flat_ids = []
  for c in fold_input['sequences']:
    cid = list(c.values())[0].get('id', '?')
    (flat_ids.extend(cid) if isinstance(cid, list) else flat_ids.append(cid))
  for ch, col in zip(flat_ids, palette):
    view.setStyle({'chain': ch}, {'cartoon': {'color': col}})

# Ligands / ions / hetero-atoms have no cartoon representation, so show them
# explicitly: sticks for ligand atoms, spheres for lone ions (e.g. MG, ZN).
ligand_ids = []
for c in fold_input['sequences']:
  if 'ligand' in c:
    lid = c['ligand']['id']
    ligand_ids.extend(lid if isinstance(lid, list) else [lid])

selectors = [{'hetflag': True}] + [{'chain': lid} for lid in ligand_ids]
for sel in selectors:
  view.addStyle(sel, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.20}})
  ion = dict(sel); ion['bonds'] = 0          # atoms with no bonds = monatomic ions
  view.addStyle(ion, {'sphere': {'colorscheme': 'greenCarbon', 'radius': 0.5}})

if show_sidechains:
  view.addStyle(
    {'and': [{'resn': ['GLY', 'PRO'], 'invert': True}, {'atom': ['C', 'O', 'N'], 'invert': True}]},
    {'stick': {'colorscheme': 'WhiteCarbon', 'radius': 0.3}},
  )
  view.addStyle({'and': [{'resn': 'GLY'}, {'atom': 'CA'}]},
                {'sphere': {'colorscheme': 'WhiteCarbon', 'radius': 0.3}})
if show_mainchains:
  view.addStyle({'atom': ['C', 'O', 'N', 'CA']},
                {'stick': {'colorscheme': 'WhiteCarbon', 'radius': 0.3}})

view.zoomTo()
view.show()


In [ ]:
#@title Quality metrics and plots
import json, os
import numpy as np
import matplotlib.pyplot as plt

conf_path = f'{OUTPUT_DIR}/{jobname}/{jobname}_confidences.json'
summ_path = f'{OUTPUT_DIR}/{jobname}/{jobname}_summary_confidences.json'

with open(conf_path) as f:
  conf = json.load(f)
with open(summ_path) as f:
  summ = json.load(f)

plddts = np.array(conf.get('atom_plddts', conf.get('token_plddts', [])), dtype=float)
plddt_chain_ids = conf.get('atom_chain_ids', conf.get('token_chain_ids', []))   # pLDDT is per-ATOM
token_chain_ids = conf.get('token_chain_ids', [])                                # PAE is per-TOKEN
pae = np.array(conf.get('pae', []), dtype=float)

# ── Summary (ipTM is None for single-chain jobs — guard before formatting) ─
def fmt(v):
  return f'{v:.3f}' if isinstance(v, (int, float)) else 'n/a'

mean_plddt = summ.get('mean_plddt')
if mean_plddt is None and plddts.size:
  mean_plddt = float(np.mean(plddts))
iptm = summ.get('iptm')

print('=' * 38)
print(f'Mean pLDDT     : {fmt(mean_plddt)}')
print(f'pTM            : {fmt(summ.get("ptm"))}')
print(f'ipTM           : {fmt(iptm)}' + ('   (single chain — no interface)' if iptm is None else ''))
print(f'Ranking score  : {fmt(summ.get("ranking_score"))}')
print('=' * 38)

# ── Plots ───────────────────────────────────────────────────
has_pae = pae.ndim == 2 and pae.size > 0
ncols = 2 if has_pae else 1
fig, axes = plt.subplots(1, ncols, figsize=(13 if has_pae else 6.5, 4))
axes = np.atleast_1d(axes)

# pLDDT per residue — a line (coloured per chain when there is more than one)
ax = axes[0]
x = np.arange(len(plddts))
xmax = max(len(plddts) - 1, 1)
ax.set_xlim(0, xmax)
ax.set_ylim(0, 100)

unique_chains = list(dict.fromkeys(plddt_chain_ids))
if len(unique_chains) > 1:
  colors = plt.cm.tab10(np.linspace(0, 1, len(unique_chains)))
  tcid = np.array(plddt_chain_ids)
  for ch, col in zip(unique_chains, colors):
    y = np.where(tcid == ch, plddts, np.nan)   # NaN gaps keep chains as separate lines
    ax.plot(x, y, lw=1.5, color=col, label=f'Chain {ch}')
  for b in [i for i in range(1, len(plddt_chain_ids)) if plddt_chain_ids[i] != plddt_chain_ids[i-1]]:
    ax.axvline(b - 0.5, color='grey', lw=0.6, alpha=0.5)
  ax.legend(loc='lower right', fontsize=8)
else:
  ax.plot(x, plddts, lw=1.5, color='#1f77b4')

for y in (50, 70, 90):
  ax.axhline(y, ls='--', lw=0.7, color='grey', alpha=0.5)
  ax.text(xmax, y, f' {y}', va='center', ha='left', fontsize=7, color='grey')
ax.set_xlabel('Atom')
ax.set_ylabel('pLDDT')
ax.set_title('Predicted pLDDT per atom')

# PAE matrix
if has_pae:
  ax = axes[1]
  im = ax.imshow(pae, cmap='bwr', vmin=0, vmax=30, interpolation='nearest')
  plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='PAE (Å)')
  if token_chain_ids:
    for b in [i for i in range(1, len(token_chain_ids)) if token_chain_ids[i] != token_chain_ids[i-1]]:
      ax.axhline(b - 0.5, c='black', lw=0.8)
      ax.axvline(b - 0.5, c='black', lw=0.8)
  ax.set_xlabel('Scored residue')
  ax.set_ylabel('Aligned residue')
  ax.set_title('Predicted Aligned Error (PAE)')

plt.tight_layout()
plt.show()


In [ ]:
#@title Download results
from google.colab import files
import os

results_zip = f'{jobname}.result.zip'
os.system(f'zip -r {results_zip} {OUTPUT_DIR}/{jobname}')
files.download(results_zip)


# Instructions <a name="Instructions"></a>

**Quick start**
1. Fill in the sequence(s) in the **Input sequence(s)** cell.
2. Press **Runtime → Run all**.
3. The install cell (first run only) downloads OpenFold3 weights and builds AF3 data files in the background — subsequent runs reuse them.

---

## Sequence input

Each molecule type has its own box. Within a box, separate multiple chains with `:`.

| Box | What goes in it | Example |
|---|---|---|
| **protein** | amino-acid sequence(s) | `MKTAY...` or `SEQ1:SEQ2` |
| **dna** | DNA sequence(s) | `CGCGAATTCGCG` |
| **rna** | RNA sequence(s) | `GCGGAUUUA` |
| **ligand_ccd** | ligand(s) by PDB CCD code | `ATP:MG:HEM` |
| **ligand_smiles** | ligand(s) by SMILES | `CC(=O)Oc1ccccc1C(=O)O` |

Chains are assigned IDs A, B, C, … following AlphaFold 3's canonical order (protein → RNA → DNA → ligand; CCD ligands before SMILES ligands). Mix freely across boxes to build a complex — e.g. a protein in **protein**, `AUGCAUGC` in **rna**, and `ATP` in **ligand_ccd**.

- **Homo-oligomers**: identical protein sequences are merged automatically, so `SEQ:SEQ` = homodimer, `SEQ:SEQ:SEQ` = homotrimer.
- Protein / DNA / RNA sequences and CCD codes are upper-cased automatically; **SMILES are left exactly as typed** (case is meaningful in SMILES).
- Spaces and newlines inside an entry are ignored, and **extra colons are forgiven** — `SEQ1::::SEQ2` is the same as `SEQ1:SEQ2`. Leave a box empty if unused.
- *Note:* because `:` separates entries, an atom-mapped SMILES that itself contains a colon (e.g. `[C:1]`) isn't supported via the box — use a raw AF3 JSON for that edge case.

## Seeds

Enter one or more model seeds in the **seeds** box, comma-separated (e.g. `1,2,3`). Each seed is an independent prediction (more seeds = more sampling, more runtime). Non-numeric characters are ignored and duplicates are dropped, so `1, 1, foo, 7` becomes seeds `1` and `7`.

## MSA modes

- **`mmseqs2_server`** *(recommended)*: queries the public [ColabFold](https://colabfold.mmseqs.com/) MMseqs2 API. Covers UniRef30 + environmental sequences for proteins. RNA/DNA chains always run MSA-free (ColabFold is protein-only).
- **`single_sequence`**: no MSA, query sequence only. Faster but less accurate, especially for monomers with close homologs.

## Output files (inside the downloaded zip)

| File | Contents |
|---|---|
| `*.cif` | Best-ranked structure in mmCIF format. B-factor = pLDDT (0–100). |
| `*_confidences.json` | Per-residue pLDDT, PAE matrix, contact probs. |
| `*_summary_confidences.json` | Mean pLDDT, pTM, ipTM, ranking score. |
| `*_ranking_scores.csv` | Ranking scores for all seed × sample combinations. |
| `seed-N_sample-M/` | Individual prediction directories (one per seed/sample). |
| `TERMS_OF_USE.md` | Apache 2.0 notice (OpenFold3 weights have no commercial restrictions). |

## Interpreting confidence scores

- **pLDDT > 90**: very high confidence.
- **pLDDT 70–90**: confident, backbone generally reliable.
- **pLDDT 50–70**: low confidence, treat with caution.
- **pLDDT < 50**: very low, likely disordered or incorrect.
- **PAE**: lower values = confident relative positioning between residue pairs. Useful for assessing interface quality in complexes.
- **ipTM > 0.8**: strong evidence for a well-defined complex interface. **ipTM is `n/a` for single-chain jobs** (there is no interface to score).

## Troubleshooting

- **Check runtime type**: `Runtime → Change runtime type → GPU` (T4 is fine; A100/L4 are faster).
- **OOM error**: reduce sequence length or use a larger-memory GPU runtime.
- **MSA server timeout**: the public ColabFold server is rate-limited. Try again later or switch to `single_sequence` mode.
- **Download popup blocked**: disable your ad blocker.
- **Weight download slow**: the OF3 checkpoint is ~1.4 GB from AWS S3; the install cell waits automatically.

## License

AlphaFold 3 source code and OpenFold3 weights are both licensed under [Apache 2.0](https://www.apache.org/licenses/LICENSE-2.0). Outputs generated with OpenFold3 weights are **not** subject to Google DeepMind's AlphaFold 3 Output Terms of Use and may be used freely, including commercially.

## Bugs / feedback

Report issues at https://github.com/sokrypton/alphafold3/issues